# API-Football Data Structure Inspection

## Objective

Inspect a real detailed fixture response collected from API-Football to identify its nested structure, available fields, and data granularity before designing the analytical database schema.

This notebook does not transform or modify the raw data.

In [1]:
import json
from pathlib import Path
from pprint import pprint

In [2]:
PROJECT_ROOT = Path.cwd().parent

DETAILS_DIR = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "api_football"
    / "fixtures"
    / "details"
)

print(DETAILS_DIR)

/Users/nathansperinde/Desktop/Portifolio/Football-Analysis/data/raw/api_football/fixtures/details


In [3]:
fixture_files = sorted(
    DETAILS_DIR.glob("fixture_*.json")
)

print(f"Detailed fixtures found: {len(fixture_files)}")

for file in fixture_files[:10]:
    print(file.name)

Detailed fixtures found: 1222
fixture_1004052.json
fixture_1004053.json
fixture_1010826.json
fixture_1010827.json
fixture_1016043.json
fixture_1016044.json
fixture_1016055.json
fixture_1016056.json
fixture_1017414.json
fixture_1017415.json


In [4]:
fixture_file = fixture_files[0]

with open(
    fixture_file,
    "r",
    encoding="utf-8",
) as file:
    fixture_data = json.load(file)

print(f"Loaded: {fixture_file.name}")

Loaded: fixture_1004052.json


In [5]:
print(type(fixture_data))
print(len(fixture_data))

<class 'list'>
1


In [6]:
print(type(fixture_data[0]))
print(fixture_data[0].keys())

<class 'dict'>
dict_keys(['fixture', 'league', 'teams', 'goals', 'score', 'events', 'lineups', 'statistics', 'players'])


In [7]:
pprint(
    fixture_data[0],
    sort_dicts=False,
)

{'fixture': {'id': 1004052,
             'referee': 'Gustavo Correia',
             'timezone': 'UTC',
             'date': '2023-04-26T19:30:00+00:00',
             'timestamp': 1682537400,
             'periods': {'first': 1682537400, 'second': 1682541000},
             'venue': {'id': 1276,
                       'name': 'Estádio Municipal 22 de Junho',
                       'city': 'Vila Nova de Famalicão'},
             'status': {'long': 'Match Finished',
                        'short': 'FT',
                        'elapsed': 90,
                        'extra': None}},
 'league': {'id': 96,
            'name': 'Taça de Portugal',
            'country': 'Portugal',
            'logo': 'https://media.api-sports.io/football/leagues/96.png',
            'flag': 'https://media.api-sports.io/flags/pt.svg',
            'season': 2022,
            'round': 'Semi-finals',
            'standings': False},
 'teams': {'home': {'id': 242,
                    'name': 'Famalicao',
         

## 2. Inspect basic fixture blocks

This section inspects the basic match-level structures contained in a detailed fixture:

- fixture;
- league;
- teams;
- goals;
- score.

The objective is to identify the available fields and nesting levels before defining the relational data model.

In [8]:
fixture = fixture_data[0]

In [9]:
basic_blocks = [
    "fixture",
    "league",
    "teams",
    "goals",
    "score",
]

for block in basic_blocks:

    print(f"\n--- {block.upper()} ---")

    pprint(
        fixture[block],
        sort_dicts=False,
    )


--- FIXTURE ---
{'id': 1004052,
 'referee': 'Gustavo Correia',
 'timezone': 'UTC',
 'date': '2023-04-26T19:30:00+00:00',
 'timestamp': 1682537400,
 'periods': {'first': 1682537400, 'second': 1682541000},
 'venue': {'id': 1276,
           'name': 'Estádio Municipal 22 de Junho',
           'city': 'Vila Nova de Famalicão'},
 'status': {'long': 'Match Finished',
            'short': 'FT',
            'elapsed': 90,
            'extra': None}}

--- LEAGUE ---
{'id': 96,
 'name': 'Taça de Portugal',
 'country': 'Portugal',
 'logo': 'https://media.api-sports.io/football/leagues/96.png',
 'flag': 'https://media.api-sports.io/flags/pt.svg',
 'season': 2022,
 'round': 'Semi-finals',
 'standings': False}

--- TEAMS ---
{'home': {'id': 242,
          'name': 'Famalicao',
          'logo': 'https://media.api-sports.io/football/teams/242.png',
          'winner': False},
 'away': {'id': 212,
          'name': 'FC Porto',
          'logo': 'https://media.api-sports.io/football/teams/212.png',
    

In [10]:
print("FIXTURE")
print(fixture["fixture"].keys())

print("\nFIXTURE > PERIODS")
print(fixture["fixture"]["periods"].keys())

print("\nFIXTURE > VENUE")
print(fixture["fixture"]["venue"].keys())

print("\nFIXTURE > STATUS")
print(fixture["fixture"]["status"].keys())

print("\nLEAGUE")
print(fixture["league"].keys())

print("\nTEAMS")
print(fixture["teams"].keys())

print("\nTEAMS > HOME")
print(fixture["teams"]["home"].keys())

print("\nTEAMS > AWAY")
print(fixture["teams"]["away"].keys())

print("\nGOALS")
print(fixture["goals"].keys())

print("\nSCORE")
print(fixture["score"].keys())

FIXTURE
dict_keys(['id', 'referee', 'timezone', 'date', 'timestamp', 'periods', 'venue', 'status'])

FIXTURE > PERIODS
dict_keys(['first', 'second'])

FIXTURE > VENUE
dict_keys(['id', 'name', 'city'])

FIXTURE > STATUS
dict_keys(['long', 'short', 'elapsed', 'extra'])

LEAGUE
dict_keys(['id', 'name', 'country', 'logo', 'flag', 'season', 'round', 'standings'])

TEAMS
dict_keys(['home', 'away'])

TEAMS > HOME
dict_keys(['id', 'name', 'logo', 'winner'])

TEAMS > AWAY
dict_keys(['id', 'name', 'logo', 'winner'])

GOALS
dict_keys(['home', 'away'])

SCORE
dict_keys(['halftime', 'fulltime', 'extratime', 'penalty'])


## 3. Inspect team match statistics

This section inspects the team-level statistics available for a single fixture.

The objective is to understand:

- the number of team records returned per match;
- the structure of each team statistics record;
- the statistical variables provided by API-Football;
- the data types and possible missing values.

No transformation is performed at this stage.

In [11]:
statistics = fixture["statistics"]

print(type(statistics))
print(f"Number of statistics records: {len(statistics)}")

<class 'list'>
Number of statistics records: 2


In [12]:
for index, item in enumerate(statistics):

    print(f"\nTEAM STATISTICS {index + 1}")

    print(type(item))
    print(item.keys())


TEAM STATISTICS 1
<class 'dict'>
dict_keys(['team', 'statistics'])

TEAM STATISTICS 2
<class 'dict'>
dict_keys(['team', 'statistics'])


In [13]:
for item in statistics:

    print(
        item["team"]["id"],
        "-",
        item["team"]["name"],
    )

242 - Famalicao
212 - FC Porto


In [14]:
print("TEAM")
print(statistics[0]["team"].keys())

print("\nSTATISTICS")
print(type(statistics[0]["statistics"]))
print(
    f"Number of statistics: "
    f"{len(statistics[0]['statistics'])}"
)

TEAM
dict_keys(['id', 'name', 'logo'])

STATISTICS
<class 'list'>
Number of statistics: 17


In [15]:
for stat in statistics[0]["statistics"]:

    print(
        f"{stat['type']}: "
        f"{stat['value']}"
    )

Shots on Goal: 3
Shots off Goal: 4
Total Shots: 8
Blocked Shots: 1
Shots insidebox: 3
Shots outsidebox: 5
Fouls: 21
Corner Kicks: 3
Offsides: 1
Ball Possession: 36%
Yellow Cards: 3
Red Cards: None
Goalkeeper Saves: 2
Total passes: 297
Passes accurate: 220
Passes %: 74%
expected_goals: None


In [16]:
for stat in statistics[0]["statistics"]:
    print(stat.keys())

dict_keys(['type', 'value'])
dict_keys(['type', 'value'])
dict_keys(['type', 'value'])
dict_keys(['type', 'value'])
dict_keys(['type', 'value'])
dict_keys(['type', 'value'])
dict_keys(['type', 'value'])
dict_keys(['type', 'value'])
dict_keys(['type', 'value'])
dict_keys(['type', 'value'])
dict_keys(['type', 'value'])
dict_keys(['type', 'value'])
dict_keys(['type', 'value'])
dict_keys(['type', 'value'])
dict_keys(['type', 'value'])
dict_keys(['type', 'value'])
dict_keys(['type', 'value'])
